# Data Foundation — worked solution

**Instructor copy. Do not hand out.** It names every defect on sight; a student who
reads this first has nothing left to discover.

Defects planted by reality, in the order students meet them:

| # | Defect | Where it bites |
|---|---|---|
| 1 | ~22% duplicate rows, up to 797 copies | Ex 1 — inflates any count or weighted mean |
| 2 | Weather station registered 3x, one per hive | Ex 2 — triples rows on a careless join, silently |
| 3 | Irregular, non-aligned sampling | Ex 3 — nothing joins on exact timestamps |
| 4 | No NULLs, but 3-6% of hours absent | Ex 3 — invisible until a complete index exists |
| 5 | UTC in a naive timestamp column | Ex 4 — local-time features come out shifted |

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

DATA = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path("workshop/data")
OUT = DATA.parent / "out"
OUT.mkdir(exist_ok=True)

beehives = pd.read_parquet(DATA / "beehives.parquet")
sensors  = pd.read_parquet(DATA / "sensors.parquet")
data     = pd.read_parquet(DATA / "data.parquet")

print(f"beehives {beehives.shape}   sensors {sensors.shape}   data {data.shape}")

## Exercise 0 — Profile

In [ ]:
print(data.head(3).to_string(), "\n")
print(data.dtypes.to_string(), "\n")
print("measurements per unit:")
print(data["measurement_unit"].value_counts().to_string(), "\n")
print("NULLs in value:", int(data["value"].isna().sum()), "  <- zero. Nothing is missing?")
print("period:", data["ts"].min(), "->", data["ts"].max())
print()
print(sensors.to_string())

`data` is long: one row per *measurement*, not per device reading. Nine sensor rows
for seven physical devices — the first hint of defect 2, if anyone reads carefully.

Zero NULLs is the trap in Exercise 3: absence here is a missing *row*, not a missing value.

## Exercise 1 — Deduplicate

In [ ]:
KEY = ["sensor_id", "measurement_unit", "ts"]

before = len(data)
distinct = len(data.drop_duplicates(subset=KEY))
print(f"rows {before:,} -> distinct keys {distinct:,}  ({100*(1-distinct/before):.1f}% redundant)")

multiplicity = data.groupby(KEY, observed=True).size()
print("worst multiplicity:", int(multiplicity.max()), "copies of a single reading")

# Do the copies disagree? This decides drop-vs-arbitrate.
conflicts = int((data.groupby(KEY, observed=True)["value"].nunique() > 1).sum())
print("keys with conflicting values:", conflicts, "-> exact copies, safe to drop")

# reading_id proves these are separate INSERTs, not a query artifact.
worst = multiplicity.idxmax()
sample = data[(data["sensor_id"] == worst[0]) & (data["measurement_unit"] == worst[1])
              & (data["ts"] == worst[2])]
print("\ndistinct reading_ids for that one reading:", sample["reading_id"].nunique())

data = data.drop_duplicates(subset=KEY)
print("after dedup:", f"{len(data):,}")

An upstream collector bug, still live and running at 20–28% every month. Note it is
*uneven* across months, so it skews any mean weighted by row count — dropping it is
not merely cosmetic.

## Exercise 2 — Reshape and join

In [ ]:
ROLE = {
    "LoRaWAN Dragino-S31-LB": "food_chamber",
    "LoRaWAN Dragino-D23-LB": "brood_chamber",
    "LoRaWAN SenseCAP-S2120": "weather",
}
sensors = sensors.assign(role=sensors["sensor_type"].map(ROLE))

# THE TRAP: one physical weather station, registered once per hive.
weather_reg = sensors[sensors["role"] == "weather"]
print("weather registrations:", len(weather_reg),
      "| distinct devices:", weather_reg["sensor_name"].nunique())
print(weather_reg[["sensor_id", "beehive_id", "sensor_name"]].to_string(index=False))

In [ ]:
data = data.merge(sensors[["sensor_id", "beehive_id", "role"]],
                  on="sensor_id", how="left", suffixes=("", "_s"))

hive_rows = data[data["role"] != "weather"]
weather_rows = data[data["role"] == "weather"]
print(f"hive rows    {len(hive_rows):>9,}")
print(f"weather rows {len(weather_rows):>9,}  <- 3x inflated")

# Collapse to the single physical station. Values are identical across the three
# registrations, so dropping duplicate (unit, ts) pairs is exact, not an average.
weather_rows = weather_rows.drop_duplicates(subset=["measurement_unit", "ts"])
print(f"weather rows {len(weather_rows):>9,}  <- collapsed")

A student who skips the collapse and merges ambient readings onto hive readings gets
three times the rows and no error. The validator's `weather shared across hives` check
exists for exactly this.

## Exercise 3 — Align time and handle gaps

In [ ]:
def to_hour(frame):
    return frame.assign(hour=frame["ts"].dt.floor("h"))

hive_h = (to_hour(hive_rows)
          .groupby(["beehive_id", "hour", "measurement_unit"], observed=True)["value"]
          .mean().unstack("measurement_unit"))
weather_h = (to_hour(weather_rows)
             .groupby(["hour", "measurement_unit"], observed=True)["value"]
             .mean().unstack("measurement_unit").add_prefix("outside_"))

# The complete grid is what makes absence visible.
hours = pd.date_range(data["ts"].min().floor("h"), data["ts"].max().floor("h"), freq="h")
grid = pd.MultiIndex.from_product([sorted(beehives["beehive_id"]), hours],
                                  names=["beehive_id", "hour"])
features = hive_h.reindex(grid)

print(f"observed hive-hours {len(hive_h):,} vs complete grid {len(features):,}"
      f"  -> {len(features) - len(hive_h):,} absent")
print("\nmissing % per column:")
print(features.isna().mean().mul(100).round(1).to_string())

In [ ]:
# How long are the gaps? The answer decides the policy.
gaps = (hive_h.reset_index().groupby("beehive_id")["hour"].diff()
        .dt.total_seconds().div(3600).dropna())
print("gap hours -- median {:.2f}, p99 {:.1f}, max {:.1f}".format(
    gaps.median(), gaps.quantile(0.99), gaps.max()))
print("gaps over 24h:", int((gaps > 24).sum()))

In [ ]:
features = features.join(weather_h, on="hour")

# Policy: interpolate gaps up to 3h; leave real outages empty rather than invent
# five days of readings. limit_area="inside" refuses to extrapolate off the ends.
MAX_GAP_H = 3
features = (features
            .groupby(level="beehive_id", group_keys=True)
            .apply(lambda g: g.droplevel("beehive_id")
                              .interpolate(method="time", limit=MAX_GAP_H,
                                           limit_area="inside")))
print("missing % after treatment:")
print(features.isna().mean().mul(100).round(1).to_string())

## Exercise 4 — Calendar, rename, validate

In [ ]:
features = features.reset_index()

# ts is UTC in a naive column. The hive is in Germany, and the period spans two
# DST changes, so localise before extracting local-time features.
local = features["hour"].dt.tz_localize("UTC").dt.tz_convert("Europe/Berlin")
features["hour_local"] = local.dt.hour
features["month"] = local.dt.month
features["dayofweek"] = local.dt.dayofweek

features = features.merge(beehives[["beehive_id", "name"]], on="beehive_id", how="left")
features = features.rename(columns={
    "name": "hive_name",
    "tempC1": "brood_temp_c1", "tempC2": "brood_temp_c2", "tempC3": "brood_temp_c3",
    "temperature": "food_temp", "relativeHumidity": "food_humidity",
    "outside_temperature": "outside_temp",
    "outside_relativeHumidity": "outside_humidity",
    "outside_windSpeed": "outside_wind_speed",
    "outside_windDirection": "outside_wind_dir",
    "outside_uvIndex": "outside_uv_index",
    "outside_pressure": "outside_pressure_pa",
    "outside_lightIntensity": "outside_light",
    "outside_rainGauge": "outside_rain",
})

assert not features.duplicated(subset=["beehive_id", "hour"]).any()
assert len(features) == features["beehive_id"].nunique() * features["hour"].nunique()

features.to_parquet(OUT / "features_hourly.parquet", index=False)
print(f"{len(features):,} rows x {features.shape[1]} cols -> {OUT / 'features_hourly.parquet'}")

```bash
uv run python workshop/validate_features.py workshop/out/features_hourly.parquet
```

Expected: all checks pass, worst column ~3.6% empty — the multi-day outages, correctly
left as gaps rather than invented.